# 第14回 A2A（Agent2Agent）

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session14/session14_a2a.ipynb)

第13回では、MCP を使ってエージェントから外部の Tool・Resource に接続した。今回は、独立したエージェントへタスクを委譲し、結果を交換する A2A を扱う。

Google ADK と LangGraph で実装したエージェントを MCP で探索し、A2A で連携させる。旅行手配の模擬処理を題材に、次の流れを確認する。

- MCP と A2A の役割の違いを説明する
- Agent Card から担当 Agent を発見し、Task を委譲する
- 追加質問への回答と、Artifact・完了状態の受信を確認する

---

## 0. 環境準備

Python 3.13 以上と Bash が必要。クローン・API キー・Dev Container の設定は [README](README.md#事前準備) を参照する。

ローカルでは `.venv/` のカーネルを選ぶ。Colab は Python 3.13 以上のカーネルを使い、`IS_COLAB` を `True` にする。リポジトリを用意し、`session14/` またはその親から以下を実行する。

Dev Container では起動時に依存が導入される。Colab では以下のセルでサーバー用 `.venv/` と実行カーネルに同じ lock の依存を導入する。サーバー起動と会話はハンズオン節で行う。Notebook と Studio は同時に使用しない。


In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
# @title 必要なパッケージをインストールする
if IS_COLAB:
    import sys
    from pathlib import Path

    if Path('session14').is_dir():
        %cd session14
    assert Path('session14_a2a.ipynb').is_file(), 'session14/ に移動してから実行する'
    if sys.version_info < (3, 13):
        raise RuntimeError('Python 3.13 以上のカーネルを使うか、Dev Container を利用する')

    %pip install -q uv
    !uv sync --locked --python {sys.executable}
    !uv export --locked --no-dev --no-hashes --no-emit-project --output-file /tmp/session14-requirements.txt > /dev/null
    %pip install -q --no-deps -r /tmp/session14-requirements.txt .


## 1. A2A の概要

Agent2Agent（A2A）プロトコルは、異なる AI エージェントフレームワーク間の通信と協調を可能にするオープン標準。LangGraph、CrewAI、Google ADK など、出自や実装方式が異なるエージェント同士の相互運用性を確保する。

### A2A の中核概念

| 要素 | 役割 |
| --- | --- |
| Agent Card | Agent の能力、接続先、対応する通信方式を公開する情報 |
| Task | 依頼された仕事とその状態。Task ID で追跡する |
| Message | Client と Server が交換する依頼・追加質問・回答 |
| Artifact | Task の実行によって生成された成果物 |
| Part | Message や Artifact を構成するテキスト・データ・ファイルなどの単位 |

#### Core Actors

A2A には3つの主要な主体が存在する。

- **User**: エージェントへの依頼を開始する利用者
- **A2A Client（Client Agent）**: 利用者に代わってアクションや情報を要求するアプリケーションまたは AI エージェント
- **A2A Server（Remote Agent）**: Client からの要求を処理して結果を返す HTTP endpoint を提供する AI エージェントまたはシステム

Remote Agent は内部実装を公開しない「opaque」なシステムとして動作する。Client は相手の内部処理を理解せずに、その公開能力を利用できる。

![a2a](./assets/a2a-overview.png)

### Agent Card

Agent Card はエージェントの能力と接続方法を示す情報であり、通常は JSON ファイルとして公開する。エージェントの名前、接続先 URL、バージョンに加え、ストリーミングや Push 通知などの対応能力、提供する Skill、標準の入出力モード、認証要件を記述する。

Card の記載だけで相手の身元や安全性を保証するものではない。

次は天気エージェント `WeatherBot` の Agent Card の例。

| 項目名 | 説明 |
| --- | --- |
| `name` / `description` | Agent の名前と役割の説明 |
| `url` | Agent を呼び出す接続先 |
| `preferredTransport` | `url` で使う優先通信方式 |
| `protocolVersion` | Agent が対応する A2A プロトコルのバージョン |
| `version` | Agent 自体のバージョン |
| `capabilities.streaming` | ストリーミング応答への対応可否 |
| `defaultInputModes` / `defaultOutputModes` | 標準で対応する入力・出力の MIME タイプ |
| `skills` | 提供する能力の一覧。各 Skill は ID や説明を持つ |
| `skills[].inputModes` / `skills[].outputModes` | Skill ごとの入出力形式。指定時は標準の形式を上書きする |

```json
{
  "name": "WeatherBot",
  "description": "Provides accurate weather forecasts and historical data.",
  "url": "http://weather-service.example.com/a2a",
  "protocolVersion": "0.3.0",
  "preferredTransport": "JSONRPC",
  "version": "1.0.0",
  "capabilities": {
    "streaming": false
  },
  "defaultInputModes": ["text/plain"],
  "defaultOutputModes": ["text/plain"],
  "skills": [
    {
      "id": "get_current_weather",
      "name": "Get Current Weather",
      "description": "Retrieve real-time weather for any location.",
      "inputModes": ["text"],
      "outputModes": ["text"],
      "examples": [
        "What’s the weather in Paris?",
        "Current conditions in Tokyo"
      ],
      "tags": ["weather", "current", "real-time"]
    },
    {
      "id": "get_forecast",
      "name": "Get Forecast",
      "description": "Get 5-day weather predictions.",
      "inputModes": ["text"],
      "outputModes": ["text"],
      "examples": [
        "5-day forecast for New York",
        "Will it rain in London this weekend?"
      ],
      "tags": ["weather", "forecast", "prediction"]
    }
  ]
}
```

### Agent の発見

Agent Card を Client が見つける主な仕組みは次の3つ。

- **Well-Known URI**: `/.well-known/agent-card.json` などの標準化されたパスで Agent Card を公開する。公開環境やドメイン内での自動探索に向く
- **Curated Registries**: Agent Card を中央カタログへ登録し、条件に基づいて検索する。集中管理とアクセス制御が必要な企業環境に向く
- **Direct Configuration**: Agent Card の情報を設定へ埋め込むか、非公開で共有する。密結合したシステムや動的探索が不要な環境に向く

Agent Card に秘密情報を含めない場合でも、公開 endpoint は保護する必要がある。特に組織内の能力情報を扱う場合は、アクセス制御、相互 TLS（mTLS）、ネットワーク制限などを適用する。

### 認証

Client は利用する Remote Agent を決定した後、Agent Card の `securitySchemes` と `security` に示された方式で認証する。A2A は API key、OAuth 2.0、OpenID Connect など、OpenAPI に準拠した認証方式を表現できる。

認証は Client の身元や credential を確認する処理であり、認証後にどの Skill や Resource の利用を許可するかは Remote Agent 側の認可処理となる。API key や access token は A2A の Message や Agent Card に含めず、HTTPS の `Authorization` header など、選択した認証方式に対応する HTTP header で送信する。

Agent Card は必要な認証方式を宣言するが、秘密情報そのものは保持しない。Client は card の宣言に従って credential を取得し、安全な通信路で提示する。

以下のハンズオンは管理下のローカルサーバーを使うため、接続先の認証・認可は実装していない。ここで説明した仕組みを演習で実装済みとは扱わない。

### 通信

A2A は用途に応じた複数の対話方式を提供する。

- **Synchronous Request/Response**: 短時間で完了する処理向け。Client は要求を送り、完全な結果が返るまで待機する
- **Asynchronous Polling**: 長時間の処理向け。Server は`working`状態とTask IDを返し、Clientは状態を確認する
- **Streaming Updates（SSE）**: Serverが状態変化や部分Artifactを逐次送る
- **Push Notifications（Webhooks）**: 非常に長いTaskの重要な状態変化をWebhookへ通知する

本教材は a2a-sdk 0.3.26（A2A 0.3系）のJSON-RPC bindingを使う。通常送信は `message/send`、ストリーミング送信は `message/stream`。1.0系のメソッド名・Message形式とは混在させない。

| 項目名 | 説明 |
| --- | --- |
| `jsonrpc` | JSON-RPC のバージョン |
| `id` | RPC の要求と応答を対応付ける ID |
| `method` | 呼び出す A2A 操作 |
| `params.message` | Agent へ送る Message |
| `params.message.messageId` | 送信者が付ける Message ごとの一意な ID |
| `params.message.role` | Message の送信側が Client か Agent かを示す |
| `params.message.parts` | 本文を構成する要素の配列。各要素は種類と内容を持つ |
| `params.configuration.acceptedOutputModes` | Client が受け取れる出力の MIME タイプ |

```json
{
  "jsonrpc": "2.0",
  "id": "request-1",
  "method": "message/send",
  "params": {
    "message": {
      "messageId": "message-1",
      "role": "user",
      "parts": [
        {
          "text": "What is the exchange rate from USD to EUR?",
          "kind": "text"
        }
      ]
    },
    "configuration": {
      "acceptedOutputModes": ["text/plain"]
    }
  }
}
```

JSON-RPCのストリーミングでは`message/stream`を使う。ServerはTask、状態更新、Artifact更新をSSE eventとして返す。

| 項目名 | 説明 |
| --- | --- |
| `id` | RPC の要求とストリーミング応答を対応付ける ID |
| `method` | 呼び出す A2A 操作 |
| `params.message.messageId` | 送信者が付ける Message ごとの一意な ID |
| `params.message.contextId` | 関連するやり取りをまとめる文脈の ID |
| `params.message.role` | Message の送信側が Client か Agent かを示す |
| `params.message.parts` | 本文を構成する要素の配列。各要素は種類と内容を持つ |

```json
{
  "jsonrpc": "2.0",
  "id": "request-2",
  "method": "message/stream",
  "params": {
    "message": {
      "messageId": "message-2",
      "contextId": "context-1",
      "role": "user",
      "parts": [
        {
          "text": "What’s the exchange rate for JPY to GBP today?",
          "kind": "text"
        }
      ]
    }
  }
}
```

追加情報や承認が必要なRemote AgentはTaskを`input-required`へ遷移させる。Clientは同じ`taskId`と`contextId`、新しい`messageId`を持つMessageを送って処理を再開する。Agent CardはstreamingやPush通知への対応を明示する。

---

## 2. A2A と MCP

A2A は Anthropic の Model Context Protocol（MCP）を置き換えるものではなく、補完するプロトコル。MCP はエージェントが外部データや Tool を利用するための接続を標準化し、A2A は独立したエージェント同士の協調、タスク委譲、情報交換を標準化する。

[![](https://mermaid.ink/img/pako:eNp9kcFqwzAMhl9F6LRBemiOOQzShkGhhbCMXZYdNEdNA0k8HIWulL77FHvNCoX5YOuXPv228BmNrRgT3Lf2aA7kBLYvZQ-6Cjs6w2nNvbyXGBR4WeIHLBZPkMapVnwq9jvkzoo1tlUimCji0VdyNcvVLKjZLKA3SHCfouX1Alj-D8YzGM_gzQQe3K1zpXY6cAtr2wt__70YHrT6OLeq8C0rZ48DO237je6IrTXUPjctDwp5AZOC4jQId3f4GxuxLlspHELISOiTBr5D03wzeU6H1jDCjl1HTYXJGeXA3fRtFe9pbAUvEdIotjj1BhNxI0c4flUknDVUO-pC8vIDuD6dQw?type=png)](https://mermaid.live/edit#pako:eNp9kcFqwzAMhl9F6LRBemiOOQzShkGhhbCMXZYdNEdNA0k8HIWulL77FHvNCoX5YOuXPv228BmNrRgT3Lf2aA7kBLYvZQ-6Cjs6w2nNvbyXGBR4WeIHLBZPkMapVnwq9jvkzoo1tlUimCji0VdyNcvVLKjZLKA3SHCfouX1Alj-D8YzGM_gzQQe3K1zpXY6cAtr2wt__70YHrT6OLeq8C0rZ48DO237je6IrTXUPjctDwp5AZOC4jQId3f4GxuxLlspHELISOiTBr5D03wzeU6H1jDCjl1HTYXJGeXA3fRtFe9pbAUvEdIotjj1BhNxI0c4flUknDVUO-pC8vIDuD6dQw)

**A2A と MCP の比較**

| 観点 | A2A | MCP |
|---|---|---|
| 接続対象 | エージェント ↔ エージェント | エージェント ↔ Tool・Resource |
| 主な目的 | 協調、委譲、成果物交換 | 外部能力と文脈の利用 |
| 相手の内部実装 | opaque なまま利用可能 | Server が公開する primitive を利用 |

A2A の目標は、複雑なマルチエージェントシステムの効率を高め、統合コストを減らし、相互運用性を促進すること。効果的な設計と実装には、中核要素と動作方式の理解が必要になる。

---

## 3. 実践的な用途

Inter-Agent Communication は、モジュール性、スケーラビリティ、専門性を備えた高度な AI システムを構築するために利用できる。

- **Multi-Framework Collaboration**: ADK、LangChain、CrewAI など、異なるフレームワークで作られた独立エージェントを連携させる。複雑なマルチエージェントシステムで、各エージェントに専門領域を担当させる
- **Automated Workflow Orchestration**: 企業内の複雑なワークフローをエージェント間の委譲と協調で進める。情報収集、分析、レポート生成をそれぞれ専門エージェントへ分担できる
- **Dynamic Information Retrieval**: エージェント同士がリアルタイム情報を要求・交換する。主エージェントが市場データ専門エージェントへ依頼し、外部 API から取得した結果を受け取る構成などに適用できる

---

## 4. ハンズオン: MCP を Agent Registry にした A2A 連携

公式 [a2a-samples](https://github.com/a2aproject/a2a-samples) の [`samples/python/agents/a2a_mcp`](https://github.com/a2aproject/a2a-samples/tree/main/samples/python/agents/a2a_mcp)（commit `6603ba3`、Apache License 2.0）を基に、LLM API を OpenAI に統一して `session14/` へ配置した。旅行の手配を題材に、MCP Server を Agent Card のレジストリとして使い、見つけた Agent へ A2A で Task を委譲する。

| Process | 実装 | Port | 役割 |
|---|---|---:|---|
| Studio UI Gateway | LangGraph | 2024 | Studio の入力を Orchestrator への A2A request に変換する |
| MCP Server | FastMCP（SSE） | 10100 | Agent Card の配布と OpenAI Embeddings による検索、旅行データ DB への問い合わせ Tool |
| Orchestrator Agent | OpenAI＋A2A SDK | 10101 | Planner の計画に沿って Task Agent を探索・呼び出し、結果を要約する |
| Planner Agent | LangGraph＋OpenAI | 10102 | 依頼を Task の列へ分解する。情報不足なら利用者へ質問する |
| Air Ticketing Agent | Google ADK＋OpenAI | 10103 | 航空券候補の検索と模擬手配 |
| Hotel Booking Agent | Google ADK＋OpenAI | 10104 | ホテル候補の検索と模擬手配 |
| Car Rental Agent | Google ADK＋OpenAI | 10105 | レンタカー候補の検索と模擬手配 |

[![](https://mermaid.ink/img/pako:eJx1kl1rwjAUhv_KIRdjg_lV3U0ZQlXoxvxC3ZUdEptjG4zNFlOlTP_7TltxHbgWenH6Ps9JcvLNQi2QucA2Sh_DmBsLw1mQAD3jZcDG2uJa6-3z2jS6nuNBX0lMbMA-oFbrnqhyggnlJiaMcW8Nt9qAF1GkINxWk15Kl8Y5Jec2FVJfBOBT5f0VfG7xyLOCGfIk8g3_jMF1mk7nCvuVjmVl8luZkmeqeJJgtX1FlS_Euboq5KJFqCcNLGS4RSuTqCLwBm8l2r6JOoS-0Akp6NEZ_YN2bqJtQvvcwIzyXN3inqpcncCAjfpTl4i9Tk2I0ICNTMSKR8VATjBaFgmYozmgKWX5PyIiSbPJSm_z6l204I42kX_af1p8pWiyFU3zgGoluOWFvoRGtIkaDHrL-4BdEvkKwqwu1gF7IDV7BLZDs-NS0MX6ZjbGXXHFBG54qiw7n38A4_u77A?type=png)](https://mermaid.live/edit#pako:eJx1kl1rwjAUhv_KIRdjg_lV3U0ZQlXoxvxC3ZUdEptjG4zNFlOlTP_7TltxHbgWenH6Ps9JcvLNQi2QucA2Sh_DmBsLw1mQAD3jZcDG2uJa6-3z2jS6nuNBX0lMbMA-oFbrnqhyggnlJiaMcW8Nt9qAF1GkINxWk15Kl8Y5Jec2FVJfBOBT5f0VfG7xyLOCGfIk8g3_jMF1mk7nCvuVjmVl8luZkmeqeJJgtX1FlS_Euboq5KJFqCcNLGS4RSuTqCLwBm8l2r6JOoS-0Akp6NEZ_YN2bqJtQvvcwIzyXN3inqpcncCAjfpTl4i9Tk2I0ICNTMSKR8VATjBaFgmYozmgKWX5PyIiSbPJSm_z6l204I42kX_af1p8pWiyFU3zgGoluOWFvoRGtIkaDHrL-4BdEvkKwqwu1gF7IDV7BLZDs-NS0MX6ZjbGXXHFBG54qiw7n38A4_u77A)

**a2a_mcp サンプルの構成**

MCP Server が公開するもの:

- **Resource** `resource://agent_cards/list`、`resource://agent_cards/{card_name}`: `agent_cards/*.json` の一覧と個別の Agent Card
- **Tool** `find_agent`: 依頼文と各 Agent Card の埋め込みの内積を取り、最も近い Agent Card を返す
- **Tool** `query_travel_data`: `travel_agency.db`（SQLite）へ `SELECT` 文を実行する。Task Agent が予約候補の検索に使う

実行の流れ:

1. Notebook は直接、Studio は UI Gateway 経由で Orchestrator へ A2A request を送る
2. Orchestrator が MCP の Resource から Planner の Agent Card を取得し、A2A で計画を依頼する
3. Planner は不足情報があれば `input-required` で質問し、揃ったら Task の一覧を Artifact として返す
4. Orchestrator は Task ごとに MCP の `find_agent` で担当 Agent を探し、A2A の streaming で委譲する
5. Task Agent は MCP の `query_travel_data` で DB を検索し、検索結果に基づく模擬応答を返す
6. Orchestrator が全結果を要約して返す

A2A の相手は起動前に固定されておらず、MCP の検索結果で決まる。Task Agent 3体は同じ `TravelAgent` クラスで、起動時に渡す Agent Card の `name` に対応するプロンプトによって役割が切り替わる（`src/a2a_mcp/agents/__main__.py`）。

### サーバーを起動する

Notebook と LangGraph Studio は同じ MCP Server と5体の A2A Agent を使う。`session14/` で次を実行し、`Ctrl+C` まで全サービスを起動したままにする。

```bash
bash run.sh --serve
```

個別に起動する場合は、MCP Server を port `10100`、Orchestrator、Planner、3体の Task Agent を port `10101`〜`10105` でそれぞれ起動する。MCP Server は `agent_cards/` と `travel_agency.db` を相対パスで読むため、必ず `session14/` をカレントディレクトリにする。

### LangGraph Studio をUIとして使う

Studio を使う場合は、別の端末で開発サーバーを起動し、表示された Studio URL を開く。Notebook の実行ではこの起動は不要。

```bash
uv run --locked langgraph dev --no-browser
```

Studio で `a2a_demo` graph を選ぶ。この graph はLLM処理を持たない薄いUI Gatewayであり、入力を port `10101` のOrchestratorへA2Aで転送する。以降のPlanner呼び出し、MCPによるAgent探索、Task Agent呼び出しはNotebook経路と同一。`input-required`が返った場合は、同じStudio threadへ回答を送って既存のA2A Taskを再開する。

### MCP Server で Agent を探索する

以下のセルでは、MCP による探索を単独で確認する。Orchestrator と同じ手順で、MCP の Resource から Agent Card の一覧を読み、`find_agent` Toolで依頼文に合うAgentを検索する。

In [ ]:
# @title MCP Server から Agent Card を探索する
import json

from a2a_mcp.mcp import client as mcp_client

async with mcp_client.init_session('localhost', 10100, 'sse') as session:
    listed = await mcp_client.find_resource(session, 'resource://agent_cards/list')
    print('Agent Card URIs:')
    for uri in json.loads(listed.contents[0].text)['agent_cards']:
        print(f'  - {uri}')

    for query in [
        'Book a round-trip flight from SFO to LHR in economy class',
        'Book a hotel for 1 adult in Central London for 2 nights',
        'Rent an SUV in London for 3 days',
    ]:
        found = await mcp_client.find_agent(session, query)
        card = json.loads(found.content[0].text)
        print(f"{query}\n  -> {card['name']} ({card['url']})")

### Agent Card を確認する

各 A2A Server は標準 endpoint `/.well-known/agent-card.json` で Agent Card を公開する。MCP Server が配布するのは同じ JSON（`agent_cards/*.json`）で、Orchestrator はそこに書かれた `url` へ A2A で接続する。

In [ ]:
# @title 起動した5体のAgent Cardを確認する
import asyncio

import httpx

AGENT_URLS = {
    'orchestrator': 'http://localhost:10101',
    'planner': 'http://localhost:10102',
    'air_ticketing': 'http://localhost:10103',
    'hotel_booking': 'http://localhost:10104',
    'car_rental': 'http://localhost:10105',
}


async def load_agent_cards() -> dict[str, dict]:
    async with httpx.AsyncClient(timeout=10) as client:
        responses = await asyncio.gather(*[
            client.get(f'{url}/.well-known/agent-card.json')
            for url in AGENT_URLS.values()
        ])
    return {
        name: response.raise_for_status().json()
        for name, response in zip(AGENT_URLS, responses, strict=True)
    }


cards = await load_agent_cards()
for key, card in cards.items():
    print(f"{key}: {card['name']} streaming={card['capabilities']['streaming']}")
    for skill in card['skills']:
        print(f"  - {skill['id']}: {skill['description']}")

### Orchestrator へ依頼する

Notebook を A2A Client として、Orchestrator へ `message/stream` で依頼を送る。Orchestrator は下流の進捗を選別し、task/context IDを自分のTaskへ対応付けて中継する。Task Agentの成果物は収集し、最後に要約する。下流の完了イベントをそのまま全体の完了として扱わない。

`contextId` は関連するやり取りの文脈を識別し、`taskId` は個々の仕事を識別する。同じ文脈に複数の Task が含まれる場合もある。

Planner が情報不足と判断し、Orchestrator も既存の文脈から回答できない場合、Task は `input-required` で止まる。その場合は最初の `Task` event で受け取った `context_id` と `task_id` を付けて回答を送り、同じ Task を再開する。

In [ ]:
# @title Orchestrator Agentへ依頼し、A2Aのeventを表示する
import json
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, A2AClient
from a2a.types import (
    MessageSendParams,
    SendStreamingMessageRequest,
    SendStreamingMessageSuccessResponse,
    Task,
    TaskArtifactUpdateEvent,
    TaskStatusUpdateEvent,
)

ORCHESTRATOR_URL = 'http://localhost:10101'


def _describe_parts(parts) -> str:
    texts = []
    for part in parts:
        root = part.root
        if root.kind == 'text':
            texts.append(root.text)
        elif root.kind == 'data':
            texts.append(json.dumps(root.data, ensure_ascii=False, indent=2))
    return '\n'.join(texts)


async def ask_orchestrator(
    text: str,
    context_id: str | None = None,
    task_id: str | None = None,
) -> dict[str, str | None]:
    message = {
        'role': 'user',
        'parts': [{'kind': 'text', 'text': text}],
        'messageId': uuid4().hex,
    }
    if context_id:
        message['contextId'] = context_id
    if task_id:
        message['taskId'] = task_id

    last_state = None
    async with httpx.AsyncClient(timeout=httpx.Timeout(180.0, connect=10.0)) as httpx_client:
        card = await A2ACardResolver(httpx_client, ORCHESTRATOR_URL).get_agent_card()
        client = A2AClient(httpx_client, card)
        request = SendStreamingMessageRequest(
            id=str(uuid4()),
            params=MessageSendParams(message=message),
        )
        async for chunk in client.send_message_streaming(request):
            if not isinstance(chunk.root, SendStreamingMessageSuccessResponse):
                print(f'[error] {chunk.root.error}')
                continue
            event = chunk.root.result
            if isinstance(event, Task):
                # Orchestrator自身のTask。再開時はこのIDを使う
                context_id, task_id = event.context_id, event.id
                print(f'[task] {task_id} ({event.status.state.value})')
            elif isinstance(event, TaskStatusUpdateEvent):
                last_state = event.status.state.value
                if event.status.message:
                    print(f'[{last_state}] {_describe_parts(event.status.message.parts)}')
            elif isinstance(event, TaskArtifactUpdateEvent):
                print(f'[artifact: {event.artifact.name}]')
                print(_describe_parts(event.artifact.parts))
    return {'context_id': context_id, 'task_id': task_id, 'state': last_state}


session = await ask_orchestrator(
    'Plan my business trip from San Francisco to London, '
    'departing June 24 2025 and returning June 28 2025, for 1 traveller.'
)
session

最後の state が `input-required` なら、表示された質問への回答を同じ `context_id`・`task_id` で送る。回答内容は質問に合わせて書き換える。

In [ ]:
# @title input-requiredになったTaskへ回答して再開する
if session['state'] == 'input-required':
    session = await ask_orchestrator(
        'Economy class, a budget hotel with a single room, total budget 5000 USD, '
        'and I need a sedan rental car.',
        context_id=session['context_id'],
        task_id=session['task_id'],
    )
session

### 観察ポイント

- Orchestrator の log で、Task ごとに MCP の `find_agent` が呼ばれ、選ばれた Agent Card の `url` へ A2A で接続する流れを確認する
- Task Agent の log で、LLM が生成した SQL が `query_travel_data` Tool 経由で `travel_agency.db` へ発行されることを確認する
- `agent_cards/` に JSON を追加し、対応する A2A Server を別ポートで起動し、MCP Server を再起動すれば探索対象を増やせる。新しい役割には対応する実装とプロンプトも必要であり、Card の追加だけで能力が増えるわけではない。Agent Cardと埋め込みは起動時に読み込む。MCP が Agent Registry として機能する点が、A2A と MCP の補完関係の具体例になる
- サンプルは A2A の仕組みを示すためのもので、Agent Card や Message を未信頼入力として検証する処理は含まない。[README のサンプルの制約](README.md#サンプルの制約)も確認する

### 結果の確認と終了

- 正常系: 5体のAgent Cardが取得でき、委譲結果の要約後に `completed` となる。内容はモデルにより変動する。
- 追加質問: `input-required` で止まり、同じTaskへの回答で再開する。
- 失敗: 下流Agentの失敗や通信切断は `failed` として扱い、成功要約へ進めない。サーバーログを確認する。
- 終了: `bash run.sh --serve` とLangGraphの各端末で `Ctrl+C`。再起動するとメモリ上のA2A Taskは失われるため、新しい会話を開始する。
- APIなしの検証: `uv run --locked python -m unittest discover -s tests -v`。

---

## 5. 適用判断

A2A は、独立したエージェントの能力を共通のインターフェースで利用するための選択肢。同じ処理を複数のエージェントへ分ければ必ず良くなるわけではない。

| 接続したい対象 | 検討する構成 |
| --- | --- |
| 同じアプリケーション内の処理 | 通常の関数呼び出しやグラフ内のノード |
| 外部の Tool・Resource | MCP |
| 独立して稼働する Agent の仕事・成果物 | A2A |

A2A を使う場合も、担当範囲、受け渡す文脈、成功・失敗の条件を設計する必要がある。通信規約だけでは、担当 Agent の選択や成果物の正しさは保証されない。

[![](https://mermaid.ink/img/pako:eNqFkE8LgkAQxb_KMmc9qDeJQOgSFEXSKTsMOuqC68o6S4T63VsT6iI4l_nz3u_wZoBcFwQxlI1-5TUaFqdb1gpX957MI4O5ZfAUvr8fr0arjkdxRtkmFbXs9OQovqPzLNxPFDvHiLSjXGIjew6c-78tlAi2uXCNC7e5aI2LVrg52sVyZ120OS54oMgolAXEA3BNav5QQSXahmHyAC3r9N3mELOx5IHtCmQ6SKwMquU4fQDmrHM5?type=png)](https://mermaid.live/edit#pako:eNqFkE8LgkAQxb_KMmc9qDeJQOgSFEXSKTsMOuqC68o6S4T63VsT6iI4l_nz3u_wZoBcFwQxlI1-5TUaFqdb1gpX957MI4O5ZfAUvr8fr0arjkdxRtkmFbXs9OQovqPzLNxPFDvHiLSjXGIjew6c-78tlAi2uXCNC7e5aI2LVrg52sVyZ120OS54oMgolAXEA3BNav5QQSXahmHyAC3r9N3mELOx5IHtCmQ6SKwMquU4fQDmrHM5)

**A2A による専門エージェントへの委譲**